# Lab 2: Simulation of Networks of Queues in ns-2

### 2006-11-13

**Abstract**: Week 8 lab. Revisit $M/M/1$ inside the ns-2 packet
simulator, then build a tandem pair and a multipath network and compare
occupancy with Kleinrock’s independence assumption. Analysis originally
in MATLAB; the Tcl scripts remain as historical listings.

$$
$$

<!-- CIP-0012 Track A: no-op outside HTML (overridden in talk-macros-html.gpp). -->
<!-- Class-test timer: no-op outside slides HTML (overridden in talk-macros-slides-html.gpp). -->
<!-- CIP-0016: mdpp expands \diagramcredit before gpp; leftover calls are empty. -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- CIP-0012 Track A: content script load. Once-guard with alphanumeric
     names stays in the surrounding ifndef wrapper (GPP ifndef limitation). -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

# How this lab sits in the course

This is the Monday of week 8, after Erlang delay and loss and (in the
2006 timetable) after the lectures on networks of queues and Jackson’s
theorem. It has three layers of dependence on earlier material.

1.  *Lab 1 and lectures 4–6.* Part 1 is again an $M/M/1$ queue, now as
    packets on a link rather than a birth-death chain on $\Delta t$.
    Arrival rate $\lambda$ and exponential packet length with rate $\mu$
    are the same pair you simulated in week 3. The theoretical mean
    occupancy is still $\rho/(1-\rho)$ with $\rho=\lambda/\mu$, provided
    the link is stable.

2.  *Little (lecture 6) and Erlang (lectures 7–10).* You already know
    that time averages need a long enough run, and that high $\rho$
    mixes slowly. That is why the sheet asks for $\lambda=1$ and then
    $\lambda=1.9$ with $\mu=2$, the same two operating points as Lab 1.

3.  *Networks of queues (lectures 14–15, week 7).* The tandem and
    multipath parts are the first time two queues share a packet.
    Kleinrock’s independence assumption — treat each link as if it saw
    Poisson arrivals with independent service — is the analytical tool
    those lectures justify. This lab asks you to *test* it on ns-2
    traces.

A Python packet-level stand-in for ns-2 is deliberately not provided
here. The Tcl in `labs/ns2/` is the 2006 artefact. Occupancy *analysis*
is rewritten in Python, as in Lab 1.

In [ ]:
import sys
import os
_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
_py = os.path.join(_root, 'python')
if os.path.isdir(_py) and _py not in sys.path:
    sys.path.insert(0, _py)
import numpy as np
import matplotlib.pyplot as plt
import npa_plots

# The 2006 lab environment

As taught, students rebooted departmental PCs into Red Hat Linux, logged
in through Gnome, opened a terminal, fetched
`http://www.dcs.shef.ac.uk/~neil/NPA/source.zip` in Netscape, and ran
`ns mm1.tcl`. Those machine-room steps are archive colour. The scripts
that mattered are in this repository as `labs/ns2/mm1.tcl` and
`labs/ns2/multipath.tcl`.

# 1. An $M/M/1$ link in ns-2

Open `labs/ns2/mm1.tcl`. At the top, `lamda` is the packet arrival rate
and `mu` parametrises an exponential packet length in kilobytes. The
topology section is marked `# Set up the network structure`: two nodes
and a simplex link $$
\texttt{set link [\$ns simplex-link \$node1 \$node2 8Kb 0ms DropTail]}.
$$ What do the arguments of `simplex-link` do? What does simplex mean?

Run `ns mm1.tcl`. Queue length is printed to the screen;
`ns mm1.tcl > qlength.dat` writes it to a file. The 2006 sheet then
loaded that file in MATLAB (`load qlength.dat`) and plotted the running
average `cumsum(qlength)./(1:length(qlength))'`. In Python, if `qlength`
is a one-dimensional array of occupancies,

In [ ]:
qlength = npa_plots.simulate_mm1(1.0, 2.0, n_iters=100_000, rng=1)
t = np.arange(1, len(qlength) + 1)
plt.plot(t, np.cumsum(qlength) / t)
plt.xlabel("step")
plt.ylabel(r"running mean occupancy")
plt.show()
print("sample mean", qlength.mean())

Run the ns-2 script (or the discrete stand-in above) for $\lambda=1$,
$\mu=2$, then again for $\lambda=1.9$, $\mu=2$. Compute the theoretical
average queue length in each case from the $M/M/1$ formula in lectures
5–6, and compare with the plotted running mean. The helper
`simulate_mm1` is *not* ns-2: it is Lab 1’s chain. Use it only as a
numerical check on the formula while the historic Tcl is what the 2006
class ran.

# 2. Tandem $M/M/1$

Extend the simulation by adding a third node and a second queue. The
topology is two links in series.

<img class="" src="https://mlatcl.github.io/npa/slides/diagrams/npa/ns2mm2Tandem.png" style="width:55%; height:auto; display:block;">

Figure: <i>Tandem pair: node 1 to node 2 to node 3, each hop an 8 Kb
DropTail link.</i>

Save `mm1.tcl` as `tandemMM1.tcl`, then:

-   After `set node2 [$ns node]`, add `set node3 [$ns node]`.

-   After the first `simplex-link`, add
    `set link2 [$ns simplex-link $node2 $node3 8Kb 0ms DropTail]`.

-   Associate a queue with the new link:
    `set qu2 [[$ns link $node2 $node3] set queue_]` and
    `$qu2 set limit_ $m`.

-   Attach the sink to `node3` instead of `node2`.

-   Monitor the new queue,
    `set qmon2 [$ns monitor-queue $node2 $node3 ""]`, and print both
    occupancies from `outputdata`.

This is the smallest network in which a packet’s length is *reused* at
the second server: service times on the two hops are proportional to the
same $L$. That correlation is exactly what Kleinrock’s independence
assumption discards. The networks-of-queues lecture (week 7) is the
justification; here, look at whether the two running means still match
isolated $M/M/1$ formulae.

# 3. Multipath and Kleinrock

A more complex architecture is prepared in `labs/ns2/multipath.tcl`.
Read the code (the duplex links are required by the multipath routing
protocol; traffic still travels one way). Draw the architecture: source
`node1`, two parallel two-hop paths to sink `node4`. The routing command
sends traffic equally down all available paths.

1.  Use Kleinrock’s independence assumption to approximate the expected
    queue length at each link.

2.  How does that approximation compare with the simulated occupancy for
    light load $\mu=2$, $\lambda=2$; moderate load $\mu=2$, $\lambda=3$;
    and heavier load $\mu=2$, $\lambda=3.8$?

3.  Reduce each link bandwidth to 4 Kb. How does average queue length
    compare with the original example?

4.  Modify the multipath script to include a direct route from `node1`
    to `node4`, and to include another node on the way to `node4`.

*End of lab.* The ns-2 scripts stay in `labs/ns2/` until a Python
packet-level equivalent is added.

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References